In [ ]:
using SparseArrays, LinearAlgebra, KrylovKit, LsqFit, CairoMakie, LaTeXStrings
plot_settings = Theme(
    Axis = (xminorgridvisible = true,
            yminorgridvisible = true,
            xminorticks = IntervalsBetween(5),
            yminorticks = IntervalsBetween(5),
    )
)
set_theme!(plot_settings)

In [ ]:
function build_H(N::Int, J::Float64, h::Float64, g::Float64)
    dim = 2^N
    I_idx = Int[]
    J_idx = Int[]
    V = Float32[]
    
    sizehint!(I_idx, dim * (N + 1))
    sizehint!(J_idx, dim * (N + 1))
    sizehint!(V, dim * (N + 1))
    
    for i in 0:(dim-1)
        diag_val = 0.0f0
        for j in 0:(N-1)
            next_j = (j + 1) % N
            
            sz_j = ((i >> j) & 1) == 1 ? 0.5f0 : -0.5f0
            sz_next = ((i >> next_j) & 1) == 1 ? 0.5f0 : -0.5f0
            
            diag_val += J * sz_j * sz_next
            diag_val += g * sz_j
            
            flip_i = i ⊻ (1 << j)
            push!(I_idx, flip_i + 1)
            push!(J_idx, i + 1)
            push!(V, Float32(h * 0.5f0))
        end
        push!(I_idx, i + 1)
        push!(J_idx, i + 1)
        push!(V, diag_val)
    end
    
    return sparse(I_idx, J_idx, V, dim, dim)
end

function init_state_plus_y(N::Int)
    dim = 2^N
    v = zeros(ComplexF64, dim)
    for i in 0:(dim-1)
        k = count_ones(i)
        v[i+1] = (1.0f0 / sqrt(2.0f0)^N) * (1im)^k
    end
    return v
end

function init_state_haar(N::Int)
    v = randn(ComplexF64, 2^N) .+ 1im .* randn(ComplexF64, 2^N)
    return v ./ norm(v)
end

function get_entanglement_entropy(v::Vector{ComplexF64}, N::Int)
    dim_half = 2^(N ÷ 2)
    v_mat = reshape(v, (dim_half, dim_half))
    S = svdvals(v_mat)
    return sum(-abs2.(S) .* log2.(abs2.(S) .+ 1e-15)) # ./ (N ÷ 2)
end

function simulate_entropy(N, J, h, g, t_max, dt)
    H = build_H(N, J, h, g)
    v = init_state_plus_y(N)
    
    times = 0.0:dt:t_max
    entropies = Float64[]
    push!(entropies, get_entanglement_entropy(v, N))
    
    for _ in 2:length(times)
        v, _ = exponentiate(H, -1im * dt, v; ishermitian=true, tol=1e-7)
        push!(entropies, get_entanglement_entropy(v, N))
    end
    
    cum_avg = cumsum(entropies) .* dt ./ max.(times, 1e-10)
    cum_avg[1] = entropies[1]
    
    return times, entropies, cum_avg
end

@.fit_model(t, p) = p[1] * (1.0 - (p[2] / t) * (1.0 - exp(-t / p[2])))

function extract_tau(times, cum_avg)
    t_data = times[2:end]
    y_data = cum_avg[2:end]
    p0 = [y_data[end], t_data[end]/2.0]
    
    fit = curve_fit(fit_model, t_data, y_data, p0)
    errors = margin_error(fit, 0.05) 
    return fit.param[2], errors[2]
end

function apply_Pauli(v::Vector{ComplexF64}, N::Int, site::Int, op::Symbol)
    v_out = zeros(ComplexF64, length(v))
    for i in 0:(length(v)-1)
        if op == :X
            flip = i ⊻ (1 << site)
            v_out[flip+1] = 0.5f0 * v[i+1]
        elseif op == :Z
            sz = ((i >> site) & 1) == 1 ? -0.5f0 : 0.5f0
            v_out[i+1] = sz * v[i+1]
        end
    end
    return v_out
end

function compute_OTOC_heatmap(N, J, h, g, t_max, dt, op::Symbol)
    H = build_H(N, J, h, g)
    times = 0.0:dt:t_max
    r_max = N ÷ 2
    
    heatmap_data = zeros(Float64, length(times), r_max + 1)
    psi_0 = init_state_haar(N)
    
    for (t_idx, t) in enumerate(times)
        psi_t, _ = exponentiate(H, -1im * t, psi_0; ishermitian=true, tol=1e-6)
        
        phi_0 = apply_Pauli(psi_0, N, 0, op)
        phi_t, _ = exponentiate(H, -1im * t, phi_0; ishermitian=true, tol=1e-6)
        
        for r in 0:r_max
            psi_prime_t = apply_Pauli(psi_t, N, r, op)
            phi_prime_t = apply_Pauli(phi_t, N, r, op)
            
            psi_prime_0, _ = exponentiate(H, 1im * t, psi_prime_t; ishermitian=true, tol=1e-6)
            phi_prime_0, _ = exponentiate(H, 1im * t, phi_prime_t; ishermitian=true, tol=1e-6)
            
            psi_final = apply_Pauli(psi_prime_0, N, 0, op)
            
            com_state = psi_final .- phi_prime_0
            heatmap_data[t_idx, r+1] = real(dot(com_state, com_state))
        end
    end
    
    return times, 0:r_max, heatmap_data
end

function analyze_h_dependence(h_vals; J=1.0, g=0., N=10, t_max=10.0, dt=0.2)
    
    taus, tau_errs = Float64[], Float64[]
    h_min, h_max = extrema(h_vals)
    cmap = Makie.to_colormap(:plasma)
    n_colors = length(cmap)  

    fig = Figure(size = (1200, 500))
    ax1 = Axis(fig[1, 1], title = "Entropy Cumulative Average vs Time", xlabel = L"t", ylabel = L"\bar{S}(t)")
    ax2 = Axis(fig[1, 3], title = "Saturation Time vs h", xlabel = L"h_x", ylabel = "τ", xscale = log10, limits = ((h_min,h_max),(0,nothing)))
    
    for h in h_vals
        times, entropies, cum_avg = simulate_entropy(N, J, h, g, t_max, dt)
        tau, err = extract_tau(times, cum_avg)
        push!(taus, tau)
        push!(tau_errs, err)
        
        norm_val = (log10(h) - log10(h_min)) / (log10(h_max) - log10(h_min))
        #norm_val = h_max == h_min ? 0.5 : (h - h_min) / (h_max - h_min)
        
        color_idx = clamp(round(Int, 1 + norm_val * (n_colors - 1)), 1, n_colors)
        lines!(ax1, times, cum_avg, color = cmap[color_idx], label = "h=$(round(h,sigdigits=3))")
    end
    hlines!(ax1, [N*(1-1/(2*log(2))),log2(N÷2),log2(N÷2)/2], color = [:cyan,:blue,:green], linestyle = :dash)
    
    cb = Colorbar(fig[1, 2], colormap = :plasma, limits = (h_min, h_max), label = L"h_X",
        scale = log10,
        width = 16, tickalign = 1.0,flipaxis = false)
    
    scatter!(ax2, h_vals, taus, color = :blue, markersize = 10)
    errorbars!(ax2, h_vals, taus, tau_errs, color = :blue, whiskerwidth = 8)
    
    display(fig)
    return taus, tau_errs
end

function analyze_J_dependence(J_vals; h=1.0, g=0., N=10, t_max=10.0, dt=0.2)
    taus, tau_errs = Float64[], Float64[]
    J_min, J_max = extrema(J_vals)
    cmap = Makie.to_colormap(:magma)
    n_colors = length(cmap)  

    fig = Figure(size = (1200, 500))
    ax1 = Axis(fig[1, 1], title = "Entropy Cumulative Average vs Time", xlabel = L"Jt", ylabel = L"\bar{S}(t)")
    ax2 = Axis(fig[1, 3], title = "Saturation Time vs J", xlabel = L"J", ylabel = "τ", xscale = log10, limits = ((J_min,J_max),(0,nothing)))
    
    for j_val in J_vals
        times, entropies, cum_avg = simulate_entropy(N, j_val, h, g, t_max ./ j_val, dt ./ j_val)
        tau, err = extract_tau(j_val .* times, cum_avg)
        push!(taus, tau)
        push!(tau_errs, err)

        norm_val = (log10(j_val) - log10(J_min)) / (log10(J_max) - log10(J_min))
        #norm_val = J_max == J_min ? 0.5 : (j_val - J_min) / (J_max - J_min)
        
        color_idx = clamp(round(Int, 1 + norm_val * (n_colors - 1)), 1, n_colors)
        lines!(ax1, j_val .* times, cum_avg, color = cmap[color_idx], label = "J=$j_val")
    end
    hlines!(ax1, [N*(1-1/(2*log(2))),log2(N÷2),log2(N÷2)/2], color = [:cyan,:blue,:green], linestyle = :dash)

    cb = Colorbar(fig[1, 2], colormap = :magma, limits = (J_min, J_max), label = L"J",
        scale = log10,
        width = 16, tickalign = 1.0,flipaxis = false)

    scatter!(ax2, J_vals, taus, color = :red, markersize = 10)
    errorbars!(ax2, J_vals, taus, tau_errs, color = :red, whiskerwidth = 8)
    
    display(fig)
    return taus, tau_errs
end

function analyze_N_dependence(N_vals; J=1.0, h=0.5, g=0., t_max=10.0, dt=0.2)
    taus, tau_errs = Float64[], Float64[]
    N_min, N_max = extrema(N_vals)
    cmap = Makie.to_colormap(:darkrainbow)
    n_colors = length(cmap)  
    
    fig = Figure(size = (1200, 500))
    ax1 = Axis(fig[1, 1], title = "Entropy Cumulative Average vs Time", xlabel = L"t/N", ylabel = L"\bar{S}(t)/ N")
    ax2 = Axis(fig[1, 3], title = "Saturation Time vs N", xlabel = L"N", ylabel = "τ", limits = ((N_min-1,N_max+1),(0, nothing)))
    
    for n_val in N_vals
        times, entropies, cum_avg = simulate_entropy(n_val, J, h, g, t_max .* n_val, dt .* n_val)
        tau, err = extract_tau(times ./ n_val, cum_avg)
        push!(taus, tau)
        push!(tau_errs, err)

        #norm_val = (log10(n_val) - log10(N_min)) / (log10(N_max) - log10(N_min))
        norm_val = N_max == N_min ? 0.5 : (n_val - N_min) / (N_max - N_min)
        
        color_idx = clamp(round(Int, 1 + norm_val * (n_colors - 1)), 1, n_colors)
        lines!(ax1, times ./ n_val, cum_avg ./ n_val, color = cmap[color_idx], label = "N=$(round(n_val,digits=2))")
    end
    axislegend(ax1, position = :rb)
    
    cb = Colorbar(fig[1, 2], colormap = :darkrainbow, limits = (N_min, N_max), label = L"N",
        #scale = log10,
        width = 16, tickalign = 1.0, flipaxis = false)

    x_coords = Float64.(N_vals)
    scatter!(ax2, x_coords, taus, color = :green, markersize = 10)
    errorbars!(ax2, x_coords, taus, tau_errs, color = :green, whiskerwidth = 8)
    
    display(fig)
    return taus, tau_errs
end

function plot_OTOC_zz(; N=10, J=1.0, h=0.5, g=0., t_max=6.0, dt=0.2)
    times, r_vals, data = compute_OTOC_heatmap(N, J, h, g, t_max, dt, :Z)
    
    fig = Figure(size = (650, 500))
    ax = Axis(fig[1, 1], title = L"C_{zz}(r,t)", xlabel = L"r", ylabel = L"Jt")
    
    hm = heatmap!(ax, r_vals, times, data', colormap = :magma)
    Colorbar(fig[1, 2], hm, label = L"C_{zz}(r,t)")
    
    display(fig)
end

function plot_OTOC_xx(; N=10, J=1.0, h=0.5, g=0., t_max=6.0, dt=0.2)
    times, r_vals, data = compute_OTOC_heatmap(N, J, h, g, t_max, dt, :X)
    
    fig = Figure(size = (650, 500))
    ax = Axis(fig[1, 1], title = L"C_{xx}(r,t)", xlabel = L"r", ylabel = L"Jt")
    
    hm = heatmap!(ax, r_vals, times, data', colormap = :magma)
    Colorbar(fig[1, 2], hm, label = L"C_{xx}(r,t)")
    
    display(fig)
end

In [ ]:
h_vals = 10 .^ (-2.:0.025:1)
analyze_h_dependence(h_vals; J=1.0, g=0.2, N=12, t_max=50.0, dt=0.5);

In [ ]:
J_vals = 10 .^ (-1:0.02:2)
analyze_J_dependence(J_vals; h=1.0, g=0.1, N=12, t_max=50.0, dt=0.5);

In [ ]:
N_vals = 6:2:20
analyze_N_dependence(N_vals; J=1.0, h=0.1, g=0., t_max=20.0, dt=0.5);

In [ ]:
plot_OTOC_zz(; N=12, J=1., h=1.25, g=0.1, t_max=20., dt=0.25)

In [ ]:
plot_OTOC_xx(; N=12, J=1., h=1.25, g=0.1, t_max=20., dt=0.25)